## Prerequisites

This notebook demonstrates how to invoke a **deployed real-time inference endpoint** on Azure for the Clinical De-identification for Romanian model. Follow the steps below in order before running this notebook.

### Deployment Sequence

1. **Acquire the Azure Application Offer** — Subscribe to the *Clinical De-identification for Romanian* offer in the [Azure Marketplace](https://marketplace.azure.com) to obtain the Solution Template.
2. **Launch guided deployment** — From the Marketplace listing, start the guided deployment flow for the solution.
3. **Supply configuration** — Complete the guided UX forms (defined by the offer's `createUiDefinition`) with the required parameters.
4. **Deploy resources** — Confirm deployment; Azure provisions the resources defined in the offer's `mainTemplate.json`.
5. **Wait for deployment to succeed** — Verify the deployment reaches a *Succeeded* state before proceeding. Note the endpoint name from the deployment output.
6. **Run this notebook** — Execute the cells below against the real-time inference endpoint provisioned by the deployment.

### Environment

- **Notebook runtime**: This notebook is intended to run in **Azure Machine Learning workspace notebooks** (e.g. via an Azure ML Studio compute instance or Studio notebooks).
- **Jupyter kernel**: Use a kernel with the `azure.ai.ml` Python package installed, for example **Python 3.10 — SDK v2** (`python310-sdkv2`).
- **Authentication**: The kernel must be authenticated to Azure so that Azure ML SDK calls succeed (e.g. `DefaultAzureCredential()`). The authenticated identity must be granted Azure RBAC permissions appropriate for reading the workspace and online endpoint configuration, and for retrieving endpoint credentials. Consult your organization's access-policy guidance and the [Azure ML endpoint credential documentation](https://learn.microsoft.com/azure/machine-learning/how-to-manage-endpoints) for details.
- **Network access**: The notebook's compute instance must be able to reach the endpoint's `scoring_uri` (no outbound firewall or VNET restrictions that block it).


## Clinical Deidentification Romanian

Deidentification is essential for safeguarding patient privacy in clinical data, including texts, PDFs, images, and DICOM files containing Protected Health Information (PHI). PHI encompasses various health-related data, including common identifiers such as name, address, birth date, and Social Security Number.

- **Model**: [ro.deid.clinical](https://nlp.johnsnowlabs.com/2023/06/17/clinical_deidentification_ro.html)
- **Model Description**: This pipeline is trained with w2v_cc_300d Romanian embeddings and can be used to deidentify PHI information from medical texts in Romanian. The PHI information will be masked and obfuscated in the resulting text. The pipeline can mask, fake or obfuscate the following entities: AGE, CITY, COUNTRY, DATE, DOCTOR, EMAIL, FAX, HOSPITAL, IDNUM, LOCATION-OTHER, MEDICALRECORD, ORGANIZATION, PATIENT, PHONE, PROFESSION, STREET, ZIP, ACCOUNT, LICENSE, PLATE


In [ ]:
ENDPOINT_NAME = "<CUSTOMER_TO_PUT_DEPLOYED_ENDPOINT_NAME>"

### Input Format

To use the model, you need to provide input in one of the following supported formats:

#### JSON Format

Provide input as JSON. We support two variations within this format:

1. **Array of Text Documents**: 
   Use an array containing multiple text documents. Each element represents a separate text document.

   ```json
   {
       "text": [
           "Text document 1",
           "Text document 2",
           ...
       ]
   }

    ```

2. **Single Text Document**:
   Provide a single text document as a string.


   ```json
    {
        "text": "Single text document"
    }
   ```

#### JSON Lines (JSONL) Format

Provide input in JSON Lines format, where each line is a JSON object representing a text document.

```
{"text": "Text document 1"}
{"text": "Text document 2"}
```

### Important Parameter

- **masking_policy**: `str`

    Users can select a masking policy to determine how sensitive entities are handled:

    Example: "**Medic : Dr. Agota EVELYN, C.N.P : 2450502264401, Data setului de analize: 25 May 2022 Varsta : 77, Nume si Prenume : BUREAN MARIA Tel: +40(235)413773, E-mail : hale@gmail.com, Licență : B004256985M, Înmatriculare : CD205113, Cont : FXHZ7170951927104999, Spitalul Pentru Ochi de Deal Drumul Oprea Nr. 972 Vaslui, 737405**"

    - **masked**: Default policy that masks entities with their type.

      -> 'Medic : Dr. `<DOCTOR>, C.N.P : `<IDNUM>`, Data setului de analize: `<DATE>` Varsta : `<AGE>`, Nume si Prenume : `<PATIENT>` Tel: `<PHONE>`, E-mail : `<EMAIL>`, Licență : `<LICENSE>`, Înmatriculare : `<PLATE>`, Cont : `<ACCOUNT>`, `<HOSPITAL>` `<STREET>` `<CITY>`, `<ZIP>`'

    - **obfuscated**: Replaces sensitive entities with random values of the same type.

      -> 'Medic : Dr. `R.t.`, C.N.P : `1527271195574`, Data setului de analize: `25 May 2022` Varsta : `68`, Nume si Prenume : `DRAGAN MIHAI` Tel: `+57(182)548668`, E-mail : `tudorsmaranda@kappa.ro`, Licență : `C775129032F`, Înmatriculare : `HM172448`, Cont : `KHHO5029180812813651`, `Centrul Medical De Evaluare Si Recuperare Pentru Copii Si Tineri Cristian Serban` `Buzias` `Aleea Voinea Aiud`, `686572`'

    - **masked_fixed_length_chars**: Masks entities with a fixed length of asterisks (\*).

      -> 'Medic : Dr. `****`, C.N.P : `****`, Data setului de analize: `****` Varsta : ****, Nume si Prenume : `****` Tel: `****`, E-mail : `****`, Licență : `****`, Înmatriculare : `****`, Cont : `****`, `****` `****` `****`, `****`'

    - **masked_with_chars**: Masks entities with asterisks (\*).

      -> 'Medic : Dr. [`**********`], C.N.P : [`***********`], Data setului de analize: [`*********`] Varsta : **, Nume si Prenume : [`**********`] Tel: [`************`], E-mail : [`************`], Licență : [`*********`], Înmatriculare : [`******`], Cont : [`******************`], [`**************************`] [`******************`] [`****`]'
    
You can specify these parameters in the input as follows:

```json
{
    "text": [
        "Text document 1",
        "Text document 2",
        ...
    ],
    "masking_policy": "masked",
}
```

> **Note:** Demonstration inputs used with this endpoint should consist of approved synthetic or properly de-identified data. Never submit real protected health information (PHI) without explicit authorization and the required safeguarding controls.


### Authentication and Endpoint Lookup

The cell below uses `DefaultAzureCredential()` to authenticate to Azure, then retrieves the endpoint's scoring URI and primary access key from the workspace configuration. If you see authentication errors, verify that you are logged into Azure Cloud Shell / your compute instance and that your identity has sufficient role assignments on the workspace.


In [5]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import requests
import json


ml_client = MLClient.from_config(DefaultAzureCredential())
endpoint_url = ml_client.online_endpoints.get(ENDPOINT_NAME).scoring_uri
key = ml_client.online_endpoints.get_keys(ENDPOINT_NAME).primary_key


def invoke_realtime_endpoint(record, content_type="application/json", accept="application/json"):
    response = requests.post(
        endpoint_url,
        headers={
            "Authorization": f"Bearer {key}",
            "Content-Type": content_type,
        },
        data=json.dumps(record) if content_type == "application/json" else record,
     
    )

    response_body = response.content.decode("utf-8")

    if accept == "application/json":
        return json.loads(response_body)
    elif accept == "application/jsonlines":
        return [json.loads(line) for line in response_body.strip().split('\n') if line]
    else:
        raise ValueError(f"Unsupported accept type: {accept}")

Found the config file in: ./config.json
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


### Sample Data (Synthetic — for demonstration only)

The following cells define sample clinical text documents used in the examples below. When using this notebook with your own data, ensure all inputs are approved synthetic or properly de-identified. Never submit real protected health information (PHI) without explicit authorization and appropriate controls.


In [9]:
docs = [
"""Dados do paciente.
Nome: Mauro.
Apelido: Gonçalves.
NISS: 26 63514095.
Endereço: Calle Miguel Benitez 90.
CÓDIGO POSTAL: 28016.
Dados de cuidados.
Data de nascimento: 03/03/1946.
País: Portugal.
Idade: 70 anos Sexo: M.
Data de admissão: 12/12/2016.
Doutor: Ignacio Navarro Cuéllar NºCol: 28 28 70973.
Relatório clínico do paciente: Paciente de 70 anos, mineiro reformado, sem alergias medicamentosas conhecidas, que apresenta como história pessoal: acidente de trabalho antigo com fracturas vertebrais e das costelas; operado por doença de Dupuytren na mão direita e iliofemoral esquerda; Diabetes Mellitus tipo II, hipercolesterolemia e hiperuricemia; alcoolismo activo, fumador de 20 cigarros / dia.
Foi encaminhado dos cuidados primários porque apresentou uma vez hematúria macroscópica pós-morte e depois microhaematúria persistente, com micturição normal.
O exame físico mostrou um bom estado geral, com abdómen e genitália normais; o exame rectal foi compatível com adenoma de próstata de grau I/IV.
A urinálise mostrou 4 glóbulos vermelhos/campo e 0-5 leucócitos/campo; o resto do sedimento estava normal.
Hemograma normal; a bioquímica mostrou glicemia de 169 mg/dl e triglicéridos de 456 mg/dl; função hepática e renal normal. PSA de 1,16 ng/ml.
A citologia da urina era repetidamente desconfiada por malignidade.
A radiografia simples abdominal mostra alterações degenerativas na coluna lombar e calcificações vasculares tanto no hipocôndrio como na pélvis.
A ecografia urológica revelou cistos corticais simples no rim direito, uma bexiga inalterada com boa capacidade e uma próstata com 30g de peso.
O IVUS mostrou normofuncionalismo renal bilateral, calcificações na silhueta renal direita e ureteres artrosados com imagens de adição no terço superior de ambos os ureteres, relacionadas com pseudodiverticulose ureteral. O cistograma mostra uma bexiga com boa capacidade, mas com paredes trabeculadas em relação à bexiga de stress. A tomografia computorizada abdominal é normal.
A cistoscopia revelou a existência de pequenos tumores na bexiga, e a ressecção transuretral foi realizada com o resultado anatomopatológico do carcinoma urotelial superficial da bexiga.
Referido por: Miguel Santos - Avenida dos Aliados, 22 Portugal E-mail: nnavcu@hotmail.com.""",

"""RELAÇÃO HOSPITALAR
NOME: Pedro Gonçalves
NHC: MVANSK92F09W408A
ENDEREÇO: Rua Burcardo 7
CÓDIGO POSTA... 
RELAÇÃO HOSPITALAR
NOME: Pedro Gonçalves
NHC: MVANSK92F09W408A
ENDEREÇO: Rua Burcardo 7
CÓDIGO POSTAL: 80139
DATA DE NASCIMENTO: 03/03/1946
IDADE: 70 anos
SEXO: Homens
E-MAIL: pgon21@tim.pt
DATA DE ADMISSÃO: 12/12/2016
DOUTORA: Eva Andrade
RELATO CLÍNICO: 70 anos, aposentado, sem alergia a medicamentos conhecida, com a seguinte história: ex-acidente de trabalho com fratura de vértebras e costelas; operado de doença de Dupuytren na mão direita e ponte ílio-femoral esquerda; diabetes tipo II, hipercolesterolemia e hiperuricemia; alcoolismo ativo, fuma 20 cigarros/dia.
Ele foi-nos encaminhado por apresentar hematúria macroscópica pós-evacuação em uma ocasião e microhematúria persistente posteriormente, com evacuação normal.
O exame físico mostrou bom estado geral, com abdome e genitais normais; o toque retal foi compatível com adenoma de próstata grau I/IV.
A urinálise mostrou 4 hemácias/campo e 0-5 leucócitos/campo; o resto do sedimento era normal.
O hemograma é normal; a bioquímica mostrou uma glicemia de 169 mg/dl e triglicerídeos 456 mg/dl; função hepática e renal são normais. PSA de 1,16 ng/ml.

DIRIGIDA A: Dr. Eva Andrade - Centro Hospitalar do Medio Ave - Avenida Dos Aliados, 56
E-MAIL: evandrade@poste.pt"""

]


sample_text = """Nome do paciente: Antonio Gonçalves
NHC: 2569870.
Endereço: Rua Das Flores 23.
Código Postal: 21754-... 
Nome do paciente: Antonio Gonçalves
NHC: 2569870.
Endereço: Rua Das Flores 23.
Código Postal: 21754-987.
Dados de cuidados.
Data de nascimento: 10/10/1963.
Idade: 23 anos 
Data de admissão: 17/06/2016.
Doutora: Maria Santos"""

### JSON

#### Example 1: masked (default-policy)

In [10]:
input_json_data = {"text": sample_text}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
response_json

{'predictions': 'Nome do paciente: <PATIENT>\nNHC: <PHONE>.\nEndereço: <ORGANIZATION> Postal: <IDNUM>-... \nNome do paciente: <PATIENT>\nNHC: <PHONE>.\nEndereço: <ORGANIZATION> Postal: <IDNUM>.\nDados de cuidados.\nData de nascimento: <DATE>.\nIdade: 23 anos \nData de admissão: <DATE>.\nDoutora: <PATIENT>'}

#### Example 2: obfuscated

In [11]:
input_json_data = {"text": sample_text, "masking_policy": "obfuscated"}
response_json = invoke_realtime_endpoint(input_json_data, content_type="application/json", accept="application/json")
response_json

{'predictions': 'Nome do paciente: Dragan Mihai\nNHC: 1290367.\nEndereço: Callcenteronline.com Srl Postal: 14625-... \nNome do paciente: Dragan Mihai\nNHC: 1290367.\nEndereço: Callcenteronline.com Srl Postal: 14625-036.\nDados de cuidados.\nData de nascimento: 10/10/1963.\nIdade: 23 anos \nData de admissão: 17/06/2016.\nDoutora: Donose Robert Dan Iulian'}

### JSON Lines

In [12]:
def create_jsonl(records, masking_policy=None):
    json_records = []

    if isinstance(records, str):
        records = [records]

    for text in records:
        record = {"text": text}

        if masking_policy is not None:
            record["masking_policy"] = masking_policy
        json_records.append(record)

    json_lines = '\n'.join(json.dumps(record,ensure_ascii=False) for record in json_records)
    return json_lines


#### Example 1: masked (default-policy)

In [13]:
input_jsonl_data = create_jsonl(docs, masking_policy="masked")
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
data


[{'predictions': 'Dados do paciente.\nNome: <AGE>.\nApelido: Gonçalves.\nNISS: 26 <IDNUM>.\nEndereço: <PATIENT> 90.\nCÓDIGO POSTAL: 28016.\nDados de cuidados.\nData de nascimento: <FAX>.\nPaís: <DOCTOR>\nIdade: <AGE> anos Sexo: <PATIENT>\nData de admissão: <DATE>.\nDoutor: <DOCTOR> Cuéllar NºCol: 28 28 70973.\nRelatório clínico do paciente: Paciente de 70 anos, mineiro reformado, sem alergias medicamentosas conhecidas, que apresenta como história pessoal: acidente de trabalho antigo com fracturas vertebrais e das costelas; operado por doença de Dupuytren na mão direita e iliofemoral esquerda; Diabetes Mellitus tipo II, hipercolesterolemia e hiperuricemia; alcoolismo activo, fumador de 20 cigarros / dia.\nFoi encaminhado dos cuidados primários porque apresentou uma vez hematúria macroscópica pós-morte e depois microhaematúria persistente, com micturição normal.\nO exame físico mostrou um bom estado geral, com abdómen e genitália normais; o exame rectal foi compatível com adenoma de prós

#### Example 2: obfuscated

In [14]:
input_jsonl_data = create_jsonl(docs, masking_policy="obfuscated")
data = invoke_realtime_endpoint(input_jsonl_data, content_type="application/jsonlines" , accept="application/jsonlines" )
data


[{'predictions': 'Dados do paciente.\nNome: Fxtep.\nApelido: Gonçalves.\nNISS: 26 98245702.\nEndereço: Dragan Daniela 90.\nCÓDIGO POSTAL: 28016.\nDados de cuidados.\nData de nascimento: 78/78/4059.\nPaís: R.t.\nIdade: 76 anos Sexo: DRAGULEASA DORINA\nData de admissão: 12/12/2016.\nDoutor: R.t. Cuéllar NºCol: 28 28 70973.\nRelatório clínico do paciente: Paciente de 70 anos, mineiro reformado, sem alergias medicamentosas conhecidas, que apresenta como história pessoal: acidente de trabalho antigo com fracturas vertebrais e das costelas; operado por doença de Dupuytren na mão direita e iliofemoral esquerda; Diabetes Mellitus tipo II, hipercolesterolemia e hiperuricemia; alcoolismo activo, fumador de 20 cigarros / dia.\nFoi encaminhado dos cuidados primários porque apresentou uma vez hematúria macroscópica pós-morte e depois microhaematúria persistente, com micturição normal.\nO exame físico mostrou um bom estado geral, com abdómen e genitália normais; o exame rectal foi compatível com ade